In [12]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import RMSE

In [4]:
Y_COL    = 'Sum of кВт'
ID_COL   = 'EIC-код_cat'
DS_COL   = 'datetime'

WEATHER_COLS = [
    'temperature_2m', 'apparent_temperature', 'dew_point_2m',
    'relative_humidity_2m', 'precipitation', 'rain', 'snowfall',
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'surface_pressure', 'wind_speed_10m', 'wind_direction_10m',
    'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation',
    'direct_normal_irradiance',
]
TIME_COLS  = ['month', 'hour', 'dow', 'season']   # numeric versions, created in prepare_nf
FUTR_EXOG  = WEATHER_COLS + TIME_COLS
STAT_EXOG  = ['lat', 'lon']                        # per-location static features

HORIZON    = 744    # ~1 month of hourly steps  (used for val/test evaluation)
INPUT_SIZE = 2 * HORIZON

In [5]:
def smallest_int_dtype(min_val, max_val, signed=True):
    if signed:
        for dtype in ['int8', 'int16', 'int32', 'int64']:
            info = np.iinfo(dtype)
            if info.min <= min_val <= max_val <= info.max:
                return dtype
    else:
        for dtype in ['uint8', 'uint16', 'uint32', 'uint64']:
            info = np.iinfo(dtype)
            if 0 <= min_val <= max_val <= info.max:
                return dtype
    return 'int64'


def optimize_df_for_memory(df):
    meta = {}
    for col in df.columns:
        s = df[col]
        unique_non_null = set(s.dropna().unique())
        if unique_non_null.issubset({0, 1, True, False}) and col == 'Група':
            df[col] = s.astype('bool')
            meta[col] = {'stored_as': 'bool', 'scale': 1}
            continue
        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dtype)
            meta[col] = {'stored_as': dtype, 'scale': 1}
            continue
        if pd.api.types.is_float_dtype(s):
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype('float32')
                meta[col] = {'stored_as': 'float32', 'scale': 1}
                continue
            decimals = non_null.astype(str).apply(
                lambda x: len(x.split('.')[1].rstrip('0')) if '.' in x else 0
            ).max()
            if decimals <= 3:
                scale  = 10 ** decimals
                scaled = np.round(s * scale)
                mn, mx = int(np.nanmin(scaled)), int(np.nanmax(scaled))
                int_dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
                if np.dtype(int_dtype).itemsize < np.dtype('float32').itemsize:
                    df[col] = scaled.astype(int_dtype)
                    meta[col] = {'stored_as': int_dtype, 'scale': scale}
                    continue
            df[col] = s.astype('float32')
            meta[col] = {'stored_as': 'float32', 'scale': 1}
    return df, meta


def add_cat_helpers(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['Month_cat']       = df['datetime'].dt.month.astype(str)
    df['Day_cat']         = df['datetime'].dt.day.astype(str)
    df['Hour_cat']        = df['datetime'].dt.hour.astype(str)
    df['day_of_week_cat'] = df['datetime'].dt.dayofweek.astype(str)
    season_map = {12: 'winter', 1: 'winter', 2: 'winter',
                   3: 'spring', 4: 'spring', 5: 'spring',
                   6: 'summer', 7: 'summer', 8: 'summer',
                   9: 'autumn', 10: 'autumn', 11: 'autumn'}
    df['season_cat'] = df['datetime'].dt.month.map(season_map)
    return df


def load_and_prepare(path):
    df = pd.read_parquet(path).reset_index(drop=True)
    df.columns = df.columns.str.replace('.', '_', regex=False)
    df, _ = optimize_df_for_memory(df)
    df = add_cat_helpers(df)
    for col in df.columns:
        if col.endswith('_cat'):
            df[col] = df[col].astype(str)
    try:
        df[Y_COL] = df[Y_COL].astype('float32')
    except Exception:
        print(f'No Y_col: {Y_COL}')
    df = df.sort_values([ID_COL, DS_COL]).reset_index(drop=True)
    try:
        df.drop(columns=['Ціна розподілу ЕЕ', 'Ціна ЕЕ', 'Money_spent'], inplace=True)
    except Exception:
        pass
    return df


def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return float(np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

In [6]:
SEASON_MAP = {'winter': 0, 'spring': 1, 'summer': 2, 'autumn': 3}


def prepare_nf(df, has_y=True):
    """Convert a raw DataFrame to neuralforecast format."""
    df = df.copy()

    # Normalise datetime: strip timezone if present
    ds = pd.to_datetime(df[DS_COL])
    if ds.dt.tz is not None:
        ds = ds.dt.tz_convert(None)
    df['ds'] = ds

    df = df.rename(columns={ID_COL: 'unique_id'})
    if has_y:
        df = df.rename(columns={Y_COL: 'y'})

    # Numeric time features (neuralforecast requires numeric exog)
    df['month']  = df['Month_cat'].astype(int)
    df['hour']   = df['Hour_cat'].astype(int)
    df['dow']    = df['day_of_week_cat'].astype(int)
    df['season'] = df['season_cat'].map(SEASON_MAP).astype(int)

    # Ensure all weather cols are float32
    for col in WEATHER_COLS:
        df[col] = df[col].astype('float32')

    keep = ['unique_id', 'ds'] + (['y'] if has_y else []) + FUTR_EXOG
    return df[keep].sort_values(['unique_id', 'ds']).reset_index(drop=True)

In [7]:
train = load_and_prepare('data/silver_money_calc/train.parquet')
val   = load_and_prepare('data/silver_money_calc/val.parquet')
test  = load_and_prepare('data/silver_money_calc/test.parquet')

train_nf = prepare_nf(train)
val_nf   = prepare_nf(val)
test_nf  = prepare_nf(test)

print('train:', train_nf.shape, ' val:', val_nf.shape, ' test:', test_nf.shape)

train: (4864324, 25)  val: (304499, 25)  test: (304152, 25)


In [8]:
# One row per unique_id with static numeric features
static_df = (
    train[[ID_COL, 'Широта', 'Довгота']]
    .drop_duplicates(ID_COL)
    .rename(columns={ID_COL: 'unique_id', 'Широта': 'lat', 'Довгота': 'lon'})
    .reset_index(drop=True)
)
static_df[['lat', 'lon']] = static_df[['lat', 'lon']].astype('float32')
print(static_df.shape)
static_df.head()

(410, 3)


,unique_id,lat,lon
0,62Z0008583037334,48.442429,22.192190
1,62Z0011230718431,51.542107,31.262997
2,62Z0096677872985,48.569660,22.346380
3,62Z0101426517156,48.567394,30.232208
4,62Z013852333354Y,48.566059,30.230684


In [15]:
def make_nhits(h):
    """Build an N-HiTS model for a given forecast horizon h."""
    return NHITS(
        h=h,
        input_size=INPUT_SIZE,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
        # Multi-scale pooling: weekly (168h) → daily (24h) → hourly (1h)
        # Captures all seasonalities present in electricity data
        # n_stacks=3,
        n_blocks=[1, 1, 1],
        mlp_units=[[512, 512], [512, 512], [512, 512]],
        n_pool_kernel_size=[168, 24, 1],
        n_freq_downsample=[168, 24, 1],
        batch_size=64,
        windows_batch_size=1024,
        learning_rate=1e-3,
        max_steps=1000,
        val_check_steps=50,
        early_stop_patience_steps=5,
        scaler_type='robust',
        loss=RMSE(),
    )

## Evaluation on Val and Test

Two separate fits:
1. Fit on `train` → predict `val`
2. Fit on `train + val` → predict `test`

In [16]:
# --- Fit on train, evaluate on val ---
nf_val = NeuralForecast(models=[make_nhits(HORIZON)], freq='h')
# val_size uses the last HORIZON steps of each series for early-stopping validation
nf_val.fit(df=train_nf, static_df=static_df, val_size=HORIZON)

Seed set to 1
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name         | Type          | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | loss         | RMSE          | 0      | train | 0    
1 | padder_train | ConstantPad1d | 0      | train | 0    
2 | scaler       | TemporalNorm  | 0      | train | 0    
3 | blocks       | ModuleList    | 32.2 M | train | 0    
---------------------------------------------------------------
32.2 M    Trainable params
0         Non-trainable params
32.2 M    Total params
128.811   Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/levkupybida/miniforge3/envs/Diploma/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

RuntimeError: Invalid buffer size: 139.13 GiB

In [ ]:
val_futr    = val_nf[['unique_id', 'ds'] + FUTR_EXOG]
val_pred_df = nf_val.predict(futr_df=val_futr)

val_merged  = val_nf[['unique_id', 'ds', 'y']].merge(val_pred_df, on=['unique_id', 'ds'])

print('Validation metrics')
print('SMAPE:', smape(val_merged['y'], val_merged['NHITS']))
print('RMSE :', rmse(val_merged['y'],  val_merged['NHITS']))
print('MAPE :', mape(val_merged['y'],  val_merged['NHITS']))

In [ ]:
# --- Fit on train + val, evaluate on test ---
train_val_nf = (
    pd.concat([train_nf, val_nf])
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

nf_test = NeuralForecast(models=[make_nhits(HORIZON)], freq='h')
nf_test.fit(df=train_val_nf, static_df=static_df, val_size=HORIZON)

In [ ]:
test_futr    = test_nf[['unique_id', 'ds'] + FUTR_EXOG]
test_pred_df = nf_test.predict(futr_df=test_futr)

test_merged  = test_nf[['unique_id', 'ds', 'y']].merge(test_pred_df, on=['unique_id', 'ds'])

print('Test metrics')
print('SMAPE:', smape(test_merged['y'], test_merged['NHITS']))
print('RMSE :', rmse(test_merged['y'],  test_merged['NHITS']))
print('MAPE :', mape(test_merged['y'],  test_merged['NHITS']))

## 6-Month Prediction on `predict_X` (Recursive)

Retrain on **all** labelled data (train + val + test) with `h=HORIZON` (1 month), then
roll forward one month at a time for 6 steps.  
Each step feeds the previous predictions back as context for the next step.

In [ ]:
predict_X_raw = load_and_prepare('data/no_y_col/with_weather_v2.parquet')
predict_nf    = prepare_nf(predict_X_raw, has_y=False)

# Compute how many hours per location we need to forecast
h_final = int(predict_nf['unique_id'].value_counts().min())
print(f'Prediction horizon (hours): {h_final}  (~{h_final/24:.1f} days)')

In [ ]:
all_data_nf = (
    pd.concat([train_nf, val_nf, test_nf])
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

# Train with h=HORIZON (1 month); recursive steps will handle the full 6-month window
nf_final = NeuralForecast(models=[make_nhits(HORIZON)], freq='h')
nf_final.fit(df=all_data_nf, static_df=static_df, val_size=HORIZON)

In [ ]:
# --- Recursive 6-month forecast: roll forward one HORIZON at a time ---
predict_futr = predict_nf[['unique_id', 'ds'] + FUTR_EXOG]

context_nf   = all_data_nf.copy()
monthly_preds = []

max_rows = predict_futr.groupby('unique_id').size().max()
n_steps  = int(np.ceil(max_rows / HORIZON))
print(f'Total steps: {n_steps}  ({HORIZON}h each)')

for step in range(n_steps):
    lo, hi = step * HORIZON, (step + 1) * HORIZON

    step_futr = (
        predict_futr
        .groupby('unique_id', group_keys=False)
        .apply(lambda g: g.iloc[lo:hi])
        .reset_index(drop=True)
    )
    if step_futr.empty:
        break

    # Predict the next HORIZON hours using the current context
    step_pred = nf_final.predict(df=context_nf, futr_df=step_futr)
    monthly_preds.append(step_pred.copy())

    # Append predicted values back into context for the next step
    new_rows = step_futr.merge(
        step_pred.rename(columns={'NHITS': 'y'})[['unique_id', 'ds', 'y']],
        on=['unique_id', 'ds'],
    )
    context_nf = (
        pd.concat([context_nf, new_rows[context_nf.columns]])
        .sort_values(['unique_id', 'ds'])
        .reset_index(drop=True)
    )
    print(f'  Step {step + 1}/{n_steps} done — context rows: {len(context_nf)}')

# Combine all steps and restore original column names
final_preds = (
    pd.concat(monthly_preds)
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
    .rename(columns={'unique_id': ID_COL, 'ds': DS_COL, 'NHITS': Y_COL})
)
print(final_preds.shape)
final_preds.head(10)

In [ ]:
import os
os.makedirs('predictions', exist_ok=True)
final_preds.to_parquet('predictions/nhits_6month.parquet', index=False)
print('Saved to predictions/nhits_6month.parquet')